# 08. Bivariate Analysis — Pairwise Relationships & Contingency Tables

Bivariate analysis explores empirical relationships between pairs of variables. The analytical technique depends entirely on the scale of the two variables:

| Scenario | Feature 1 | Feature 2 | Core Techniques |
| :--- | :--- | :--- | :--- |
| **Numeric vs. Numeric** | Continuous | Continuous | Scatter plot, Pearson $r$, Spearman $\rho$, Hexbin plot |
| **Numeric vs. Categorical** | Continuous | Categorical | GroupBy aggregation, Box plots, Violin plots, Two-sample t-test |
| **Categorical vs. Categorical** | Categorical | Categorical | Contingency cross-tabulation, Stacked bar charts, Chi-Square test |

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n = 1500

education = np.random.choice(['High School', 'Bachelor', 'Graduate'], size=n, p=[0.35, 0.45, 0.20])
experience = np.random.uniform(1, 25, size=n).round(1)

# Base salary dependent on education level and years of experience
base_sal = {'High School': 35000, 'Bachelor': 52000, 'Graduate': 70000}
salary = np.array([base_sal[e] for e in education]) + (experience * 2400) + np.random.normal(0, 6000, size=n)

# Churn indicator dependent on salary
churn_prob = 1.0 / (1.0 + np.exp((salary - 65000) / 12000))
churn = (np.random.rand(n) < churn_prob).astype(int)

df_biv = pd.DataFrame({
    'education': education,
    'experience': experience,
    'salary': salary.round(2),
    'churn': churn
})
print("Bivariate Dataset Initialized. Sample:")
print(df_biv.head())

Bivariate Dataset Initialized. Sample:
     education  experience      salary  churn
0     Bachelor       8.400  77,142.150      0
1     Graduate      12.100  99,410.200      0
2  High School       3.200  42,110.800      1
3     Bachelor      18.900  96,215.400      0
4     Bachelor       5.100  63,120.900      1

## 1. Continuous vs. Continuous: Correlation & Regression Slope

In [1]:
r_pearson, p_pearson = stats.pearsonr(df_biv['experience'], df_biv['salary'])
rho_spearman, p_spearman = stats.spearmanr(df_biv['experience'], df_biv['salary'])

print("=== NUMERIC vs NUMERIC RELATIONSHIP ===")
print(f"Feature Pair: Experience vs Salary")
print(f"  Pearson Correlation Coefficient:  r = {r_pearson:+.3f} (p-value: {p_pearson:.2e})")
print(f"  Spearman Rank Correlation:        rho = {rho_spearman:+.3f} (p-value: {p_spearman:.2e})")
slope, intercept, _, _, _ = stats.linregress(df_biv['experience'], df_biv['salary'])
print(f"  Fitted Linear Equation:           Salary = ${intercept:,.2f} + (${slope:,.2f} * Experience)")

=== NUMERIC vs NUMERIC RELATIONSHIP ===
Feature Pair: Experience vs Salary
  Pearson Correlation Coefficient:  r = +0.762 (p-value: 1.14e-289)
  Spearman Rank Correlation:        rho = +0.748 (p-value: 3.42e-272)
  Fitted Linear Equation:           Salary = $48,212.45 + ($2,394.18 * Experience)

## 2. Continuous vs. Categorical: Group Differences

In [1]:
salary_by_edu = df_biv.groupby('education')['salary'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print("=== NUMERIC vs CATEGORICAL RELATIONSHIP ===")
print("Salary Distribution Partitioned by Education Level:")
print(salary_by_edu.sort_values(by='mean', ascending=False))

=== NUMERIC vs CATEGORICAL RELATIONSHIP ===
Salary Distribution Partitioned by Education Level:
             count        mean      median        std        min         max
education                                                                   
Graduate       298 100,812.450 101,210.100 16,412.300 68,140.200 134,810.500
Bachelor       682  82,410.120  81,950.400 16,890.150 51,240.100 119,410.800
High School    520  65,120.900  64,980.250 16,110.400 34,120.000  98,450.100

## 3. Categorical vs. Categorical: Cross-Tabulation & Contingency

In [1]:
ct = pd.crosstab(df_biv['education'], df_biv['churn'], margins=True, margins_name='Total')
ct_pct = pd.crosstab(df_biv['education'], df_biv['churn'], normalize='index') * 100

print("=== CATEGORICAL vs CATEGORICAL RELATIONSHIP ===")
print("Contingency Table (Absolute Frequency):")
print(ct)
print("\nRow Percentages (% Churned by Education):")
print(ct_pct.round(2).rename(columns={0: 'Retained %', 1: 'Churned %'}))

=== CATEGORICAL vs CATEGORICAL RELATIONSHIP ===
Contingency Table (Absolute Frequency):
churn          0    1  Total
education                   
Bachelor     550  132    682
Graduate     275   23    298
High School  310  210    520
Total       1135  365   1500

Row Percentages (% Churned by Education):
churn        Retained %  Churned %
education                         
Bachelor         80.650     19.350
Graduate         92.280      7.720
High School      59.620     40.380